# 02 — Universe Membership & Price Data

This notebook builds three things:
1. **Point-in-time universe membership** for SP500, SP1500, and RU3K — so we only trade stocks that were actually in the index on the trade date (avoids survivorship bias)
2. **Adjusted close prices** for every ticker in the signal file, fetched from yfinance and cached per-ticker to disk
3. **Forward returns** (1, 3, 5, 10, 20 trading days) with correct AMC/BMO entry timing, saved as an augmented parquet

**Look-ahead audit touchpoints in this notebook:**
- Universe membership is always the set known *before* the trade date (no future additions)
- Forward returns are computed *from* prices, never fed back as model inputs
- Entry date is derived from `MOSTIMPORTANTDATEUTC` hour, not from the return series

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import bisect
import time
import json
from pathlib import Path
from datetime import timedelta
import os, warnings
warnings.filterwarnings('ignore')

PROJECT       = Path(os.getenv("ATC_PROJECT_ROOT",
                               Path.cwd().parent if Path.cwd().name == 'notebooks'
                               else Path.cwd())).resolve()
DATA_DIR      = PROJECT / 'data'
UNIV_DIR      = DATA_DIR / 'universe'
PRICE_DIR     = DATA_DIR / 'prices'
SIGNALS_PQ    = DATA_DIR / 'signals.parquet'
AUGMENTED_PQ  = DATA_DIR / 'signals_with_returns.parquet'

# Load the Total slice only — we only need one row per call to compute returns
df_total = pd.read_parquet(SIGNALS_PQ, filters=[('SignalType', '=', 'Total')])
print(f'Total-slice rows: {len(df_total):,}  |  unique tickers: {df_total["BESTTICKER"].nunique():,}')


Total-slice rows: 376,790  |  unique tickers: 17,636


## 2.1 Entry Date: AMC vs BMO

The handout rule: parse the **hour (UTC)** of `MOSTIMPORTANTDATEUTC`.
- **Hour ≥ 16 UTC** → after-market-close → enter at **next** trading day's close  
- **Hour < 13 UTC** → before-market-open → enter at **same** trading day's close  
- **Hour 13–15 UTC** → ambiguous; we conservatively use next day (document this choice)

We store `entry_date` (the calendar date of entry) rather than the open price to avoid any bid/ask or intraday look-ahead. The actual price used is the **close** on `entry_date`.

In [2]:
import pandas_market_calendars as mcal

# Fall back to simple business-day logic if the library isn't installed
try:
    import pandas_market_calendars as mcal
    nyse = mcal.get_calendar('NYSE')
    USE_MARKET_CAL = True
    print('Using NYSE market calendar for trading day offsets')
except ImportError:
    USE_MARKET_CAL = False
    print('pandas_market_calendars not found — using BDay offset (close enough for close-to-close returns)')


def next_trading_day(dt: pd.Timestamp) -> pd.Timestamp:
    """Return the next NYSE trading day after dt (always tz-naive)."""
    if USE_MARKET_CAL:
        sessions = nyse.valid_days(start_date=dt + pd.Timedelta(days=1),
                                   end_date=dt + pd.Timedelta(days=10))
        result = sessions[0] if len(sessions) else dt + pd.offsets.BDay(1)
        # valid_days returns tz-aware (UTC); strip to keep consistent with tz-naive inputs
        if hasattr(result, 'tzinfo') and result.tzinfo is not None:
            result = result.tz_localize(None)
        return result
    return dt + pd.offsets.BDay(1)


def assign_entry_date(row) -> pd.Timestamp:
    call_dt = row['MOSTIMPORTANTDATEUTC']
    if pd.isna(call_dt):
        # Fall back to DocDate + 1 if timestamp is missing
        return row['DocDate'] + pd.offsets.BDay(1)
    call_local = call_dt  # already UTC
    hour = call_local.hour
    if hour < 13:          # BMO — same-day close
        return call_local.normalize().tz_localize(None)
    else:                  # AMC (≥16) or intraday (13-15) — next-day close
        return next_trading_day(call_local.normalize().tz_localize(None))


df_total = df_total.copy()
df_total['entry_date'] = df_total.apply(assign_entry_date, axis=1)
df_total['entry_date'] = pd.to_datetime(df_total['entry_date'])

# Spot-check timing distribution
hour_col = df_total['MOSTIMPORTANTDATEUTC'].dt.hour
print('Hour distribution of call timestamps (UTC):')
print(hour_col.value_counts().sort_index().to_string())
print(f"\nEntry-date offset examples (first 5 rows):")
print(df_total[['DocDate','MOSTIMPORTANTDATEUTC','entry_date']].head())


Using NYSE market calendar for trading day offsets


Hour distribution of call timestamps (UTC):
MOSTIMPORTANTDATEUTC
0.0     10640
1.0      3978
2.0      1627
3.0      1136
4.0      1829
5.0      3080
6.0      7623
7.0     13447
8.0     18126
9.0     13757
10.0     9901
11.0     7971
12.0    41922
13.0    47058
14.0    44619
15.0    40253
16.0    17135
17.0     6941
18.0     4791
19.0     2360
20.0    22314
21.0    37210
22.0    15700
23.0     3312

Entry-date offset examples (first 5 rows):
     DocDate      MOSTIMPORTANTDATEUTC entry_date
0 2010-01-04 2010-01-04 22:00:00+00:00 2010-01-05
1 2010-01-05 2010-01-05 15:00:00+00:00 2010-01-06
2 2010-01-05 2010-01-05 21:30:00+00:00 2010-01-06
3 2010-01-05 2010-01-05 22:00:00+00:00 2010-01-06
4 2010-01-05 2010-01-05 22:00:00+00:00 2010-01-06


## 2.2 Universe Membership — WRDS/Compustat + CRSP (Exact PIT)

**Source:** WRDS with institutional subscription.

- **SP500, SP400, SP600** → `comp.idxcst_his` keyed by GVKEY (our signal data already has GVKEY).
  This is the official Compustat record of S&P index membership with exact addition/removal dates.
  No reconstruction from changes needed — the intervals are stored directly.

- **Russell 3000** → built from `crsp.msf` (CRSP monthly market cap) at each annual
  June reconstitution date, filtered to eligible US common stocks (shrcd 10/11, exchcd 1/2/3).
  PERMNOs mapped to GVKEY via `crsp.ccmxpf_lnkhist`. This replaces the previous
  "all US ATC tickers" proxy with actual CRSP market-cap ranked Russell 3000 membership.

All data is cached to `data/universe/` on first run and reused thereafter.

In [3]:
import wrds

SP_CACHE   = UNIV_DIR / 'sp_constituents_wrds.parquet'
RU3K_CACHE = UNIV_DIR / 'ru3k_constituents_crsp.parquet'
LINK_CACHE = UNIV_DIR / 'crsp_compustat_link.parquet'

if SP_CACHE.exists() and RU3K_CACHE.exists():
    sp_hist   = pd.read_parquet(SP_CACHE)
    ru3k_hist = pd.read_parquet(RU3K_CACHE)
    print(f'Loaded from cache:')
    print(f'  SP constituent rows : {len(sp_hist):,}')
    print(f'  Russell 3000 rows   : {len(ru3k_hist):,}')
else:
    db = wrds.Connection()   # prompts for credentials once per session

    # ── S&P 500 / 400 / 600 — Compustat historical index constituents ─────────
    # "from" is a reserved SQL keyword in Python; alias it.
    sp_sql = """
        SELECT
            gvkey::bigint          AS gvkey,
            UPPER(indextype)       AS idx_type,
            "from"::date           AS start_dt,
            COALESCE(thru::date, '2035-12-31'::date) AS end_dt
        FROM comp.idxcst_his
        WHERE UPPER(indextype) IN ('SP500', 'SP400', 'SP600')
        ORDER BY gvkey, idx_type, start_dt
    """
    sp_hist = db.raw_sql(sp_sql, date_cols=['start_dt', 'end_dt'])
    sp_hist.to_parquet(SP_CACHE, index=False)
    print('SP constituent records fetched:')
    print(sp_hist.groupby('idx_type').size().to_string())

    # ── CRSP monthly market cap for Russell 3000 construction ─────────────────
    # Eligible = US common shares on NYSE / AMEX / NASDAQ, price > $1
    me_sql = """
        SELECT msf.permno,
               msf.date,
               ABS(msf.prc) * msf.shrout AS me
        FROM crsp.msf AS msf
        JOIN crsp.msenames AS names
          ON msf.permno = names.permno
         AND msf.date BETWEEN names.namedt AND names.nameendt
        WHERE msf.date >= '2009-01-01'
          AND names.shrcd  IN (10, 11)
          AND names.exchcd IN (1, 2, 3)
          AND ABS(msf.prc) > 1.0
          AND msf.shrout  > 0
        ORDER BY msf.date, msf.permno
    """
    crsp_me = db.raw_sql(me_sql, date_cols=['date'])
    print(f'\nCRSP market cap rows: {len(crsp_me):,}')

    # ── CRSP–Compustat link (PERMNO → GVKEY) ─────────────────────────────────
    link_sql = """
        SELECT
            lpermno::bigint AS permno,
            gvkey::bigint   AS gvkey,
            linkdt::date                              AS link_start,
            COALESCE(linkenddt::date, '2035-12-31'::date) AS link_end
        FROM crsp.ccmxpf_lnkhist
        WHERE linktype IN ('LC', 'LU', 'LS')
          AND linkprim IN ('P', 'J', 'C')
        ORDER BY permno, link_start
    """
    crsp_link = db.raw_sql(link_sql, date_cols=['link_start', 'link_end'])
    crsp_link.to_parquet(LINK_CACHE, index=False)
    print(f'CRSP–Compustat link rows: {len(crsp_link):,}')

    db.close()

    # ── Build Russell 3000 PIT membership ────────────────────────────────────
    # At each annual June reconstitution (last Friday of June), rank US common stocks
    # by CRSP market cap, take top 3000. Map PERMNO → GVKEY via the linking table.
    def last_friday_june(year):
        d = pd.Timestamp(f'{year}-06-30')
        return d - pd.Timedelta(days=(d.weekday() - 4) % 7)

    recon_dates = [last_friday_june(y) for y in range(2009, pd.Timestamp.now().year + 2)]
    crsp_me['month'] = crsp_me['date'].dt.to_period('M')

    ru3k_records = []
    for i, recon in enumerate(recon_dates):
        recon_month = recon.to_period('M')
        month_me    = crsp_me[crsp_me['month'] == recon_month]
        if month_me.empty:
            month_me = crsp_me[crsp_me['month'] == (recon_month - 1)]
        if month_me.empty:
            continue

        top3k_permnos = set(month_me.nlargest(3000, 'me')['permno'])

        link_at_recon = crsp_link[
            (crsp_link['link_start'] <= recon) & (crsp_link['link_end'] >= recon)
        ][['permno', 'gvkey']].drop_duplicates('permno')
        mapped = link_at_recon[link_at_recon['permno'].isin(top3k_permnos)]

        end_date = (recon_dates[i + 1] - pd.Timedelta(days=1)
                    if i + 1 < len(recon_dates) else pd.Timestamp('2035-12-31'))

        for _, row in mapped.iterrows():
            ru3k_records.append({
                'gvkey':    int(row['gvkey']),
                'start_dt': recon,
                'end_dt':   end_date,
            })

    ru3k_hist = pd.DataFrame(ru3k_records)
    ru3k_hist.to_parquet(RU3K_CACHE, index=False)
    print(f'\nRussell 3000 PIT records: {len(ru3k_hist):,}')
    print(f'Unique GVKEYs in Russell 3000: {ru3k_hist["gvkey"].nunique():,}')

# Ensure correct dtypes after load
for df in [sp_hist, ru3k_hist]:
    df['start_dt'] = pd.to_datetime(df['start_dt'])
    df['end_dt']   = pd.to_datetime(df['end_dt'])
    df['gvkey']    = df['gvkey'].astype('Int64')
sp_hist['idx_type'] = sp_hist['idx_type'].str.upper()

print('\nData ready:')
for idx in ['SP500', 'SP400', 'SP600']:
    sub = sp_hist[sp_hist['idx_type'] == idx]
    print(f'  {idx:6s}: {sub["gvkey"].nunique():,} unique GVKEYs, '
          f'{sub["start_dt"].min().year}–{sub["end_dt"].max().year}')
print(f'  RU3K : {ru3k_hist["gvkey"].nunique():,} unique GVKEYs, '
      f'{ru3k_hist["start_dt"].min().year}–{ru3k_hist["end_dt"].max().year}')


Loaded from cache:
  SP constituent rows : 1,505
  Russell 3000 rows   : 47,901

Data ready:
  SP500 : 500 unique GVKEYs, 1964–2035
  SP400 : 400 unique GVKEYs, 1991–2035
  SP600 : 600 unique GVKEYs, 1994–2035
  RU3K : 6,626 unique GVKEYs, 2009–2025


## 2.3 Universe Membership — GVKEY-Based Interval Lookup

No reconstruction needed. The WRDS data (`comp.idxcst_his` and the CRSP-derived Russell table)
already stores membership as `(gvkey, start_dt, end_dt)` intervals.

For any signal event with GVKEY G and entry date D:
- `in_sp500  = any row where gvkey=G, idx_type='SP500', start_dt ≤ D ≤ end_dt`
- `in_sp1500 = same but idx_type ∈ {SP500, SP400, SP600}`
- `in_ru3k   = any row where gvkey=G, start_dt ≤ D ≤ end_dt` (from CRSP reconstitution table)

Implementation: merge signals with the interval table on GVKEY, then filter on date overlap.

In [4]:
def flag_from_intervals(signals_df, hist_df, flag_col, idx_types=None):
    """
    Vectorised interval join: for each (gvkey, entry_date) in signals_df,
    check if that gvkey was a member of any idx_type in idx_types on entry_date.
    
    Returns a Series indexed like signals_df with True/False values.
    """
    h = hist_df.copy()
    if idx_types:
        h = h[h['idx_type'].isin(idx_types)]

    # Only consider GVKEYs that appear in both datasets
    h = h[h['gvkey'].isin(signals_df['gvkey_int'].dropna().unique())]

    # Merge on gvkey: each signal row gets one row per membership interval
    merged = signals_df[['gvkey_int', 'entry_date']].merge(
        h[['gvkey', 'start_dt', 'end_dt']],
        left_on='gvkey_int', right_on='gvkey',
        how='left'
    )
    # Date falls in interval?
    merged[flag_col] = (
        (merged['start_dt'] <= merged['entry_date']) &
        (merged['end_dt']   >= merged['entry_date'])
    ).fillna(False)

    # Aggregate: any matching interval → True
    result = merged.groupby(merged.index)[flag_col].any()
    return result

print('Flag helper defined.')


Flag helper defined.


## 2.4 Assign Universe Flags

Use GVKEY for the lookup — stable identifier across time, no ticker-change issues.

In [5]:
# Prepare GVKEY column from signals (stored as float64 to handle NaN; convert to Int64)
df_total['gvkey_int'] = df_total['GVKEY'].where(df_total['GVKEY'].notna()).astype('Int64')

print(f'Signal rows with valid GVKEY: {df_total["gvkey_int"].notna().sum():,} / {len(df_total):,}')
print(f'Unique GVKEYs in signal data: {df_total["gvkey_int"].dropna().nunique():,}')
print(f'Unique GVKEYs in SP table   : {sp_hist["gvkey"].nunique():,}')
print(f'Unique GVKEYs in R3K table  : {ru3k_hist["gvkey"].nunique():,}')


Signal rows with valid GVKEY: 376,790 / 376,790
Unique GVKEYs in signal data: 17,429
Unique GVKEYs in SP table   : 1,500
Unique GVKEYs in R3K table  : 6,626


In [6]:
print('Assigning universe flags via GVKEY interval lookup...')

# Assign flags directly on df_total (index-aligned)
df_total['in_sp500']  = flag_from_intervals(df_total, sp_hist,   'in_sp500',  ['SP500'])
df_total['in_sp1500'] = flag_from_intervals(df_total, sp_hist,   'in_sp1500', ['SP500','SP400','SP600'])
df_total['in_ru3k']   = flag_from_intervals(df_total, ru3k_hist, 'in_ru3k')

# Rows with no GVKEY cannot be matched — explicitly mark False
no_gvkey = df_total['gvkey_int'].isna()
for col in ['in_sp500', 'in_sp1500', 'in_ru3k']:
    df_total.loc[no_gvkey, col] = False

print('Universe coverage (exact WRDS/CRSP PIT, no approximations):')
for col in ['in_sp500', 'in_sp1500', 'in_ru3k']:
    n   = df_total[col].sum()
    pct = df_total[col].mean()
    tks = df_total.loc[df_total[col], 'BESTTICKER'].nunique()
    print(f'  {col}: {n:,} events / {tks:,} unique tickers ({pct:.1%})')


Assigning universe flags via GVKEY interval lookup...


Universe coverage (exact WRDS/CRSP PIT, no approximations):
  in_sp500: 24,041 events / 8,309 unique tickers (6.4%)
  in_sp1500: 49,809 events / 11,444 unique tickers (13.2%)
  in_ru3k: 29,538 events / 10,995 unique tickers (7.8%)


## 2.5 Price Data: yfinance with Per-Ticker Parquet Cache

Adjusted close prices fetched from yfinance, cached per-ticker.
We only fetch tickers that appear in at least one of the three universes
(identified using the GVKEY-based flags above).

In [7]:
# Universe flags are already assigned in §2.4 via GVKEY interval lookup.
# This cell is a verification / coverage check only.
print('=== Universe coverage verification ===')
for col in ['in_sp500', 'in_sp1500', 'in_ru3k']:
    sub = df_total[df_total[col]]
    yr_counts = sub.groupby(sub['entry_date'].dt.year).size()
    print(f'\n{col.upper()} — {len(sub):,} events, {sub["BESTTICKER"].nunique():,} tickers')
    print(f'  Date range: {sub["entry_date"].min().date()} → {sub["entry_date"].max().date()}')
    print(f'  Events/year (first 5): {dict(list(yr_counts.items())[:5])}')


=== Universe coverage verification ===



IN_SP500 — 24,041 events, 8,309 tickers
  Date range: 2010-01-08 → 2026-04-22
  Events/year (first 5): {2010: 1086, 2011: 1131, 2012: 1165, 2013: 1208, 2014: 1250}

IN_SP1500 — 49,809 events, 11,444 tickers
  Date range: 2010-01-07 → 2026-04-22
  Events/year (first 5): {2010: 1706, 2011: 1832, 2012: 1924, 2013: 2010, 2014: 2122}

IN_RU3K — 29,538 events, 10,995 tickers
  Date range: 2010-01-06 → 2026-04-22
  Events/year (first 5): {2010: 1138, 2011: 1471, 2012: 1544, 2013: 1417, 2014: 1446}


## 2.6 Price Data: yfinance with Per-Ticker Parquet Cache

We fetch **adjusted close prices** (split- and dividend-adjusted) from yfinance for every unique ticker. Each ticker is cached as a small Parquet file so we never re-hit the API. Delisted or unavailable tickers are logged to `data/failed_tickers.txt`.

Rule: use `BESTTICKER` as the join key (cleanest field per the handout). For delisted tickers yfinance often returns empty data — those trades will be skipped in the backtest.

In [8]:
FAILED_FILE = DATA_DIR / 'failed_tickers.txt'
already_failed = set()
if FAILED_FILE.exists():
    already_failed = set(FAILED_FILE.read_text().strip().splitlines())

# Only fetch prices for tickers in our three universes (~1.5k vs 17k total).
# Tickers not in any universe are never traded in the backtest, so their prices are unused.
universe_tickers = sorted(set(
    df_total[df_total['in_sp500'] | df_total['in_sp1500'] | df_total['in_ru3k']]['BESTTICKER'].dropna()
))
need_to_fetch = [
    t for t in universe_tickers
    if not (PRICE_DIR / f'{t}.parquet').exists() and t not in already_failed
]
print(f'Universe tickers : {len(universe_tickers):,}')
print(f'Already cached   : {len(universe_tickers) - len(need_to_fetch):,}')
print(f'To fetch         : {len(need_to_fetch):,}')


Universe tickers : 14,104
Already cached   : 6,433
To fetch         : 7,671


In [9]:
# Fetch in small batches. yfinance 1.2.x always returns MultiIndex columns (price_type, ticker).
BATCH_SIZE   = 10   # smaller batches to stay under rate limits
new_failures = []

for i in range(0, len(need_to_fetch), BATCH_SIZE):
    batch = need_to_fetch[i:i + BATCH_SIZE]
    try:
        raw = yf.download(
            tickers     = batch,
            start       = '2009-01-01',
            end         = '2026-05-01',
            auto_adjust = True,
            progress    = False,
            threads     = True,
        )
    except Exception:
        new_failures.extend(batch)
        time.sleep(2)
        continue

    if raw.empty:
        new_failures.extend(batch)
        continue

    # raw['Close'] is always a DataFrame in yfinance 1.2.x (MultiIndex level-1 = ticker)
    try:
        close_df = raw['Close']
    except KeyError:
        new_failures.extend(batch)
        continue

    for t in batch:
        try:
            if isinstance(close_df, pd.DataFrame):
                closes = close_df[t].dropna() if t in close_df.columns else pd.Series(dtype=float)
            else:  # old single-ticker format: Series
                closes = close_df.dropna()

            if closes.empty:
                new_failures.append(t)
                continue

            result = pd.DataFrame({'date': closes.index, 'close': closes.values})
            # Strip timezone from date index (yfinance 1.2.x may return tz-aware dates)
            result['date'] = pd.to_datetime(result['date']).dt.tz_localize(None)
            result.to_parquet(PRICE_DIR / f'{t}.parquet', index=False)
        except Exception:
            new_failures.append(t)

    if i > 0 and (i // BATCH_SIZE) % 20 == 0:
        cached = len(list(PRICE_DIR.glob('*.parquet')))
        print(f'  {i:,}/{len(need_to_fetch):,} processed | cached: {cached:,} | failed: {len(new_failures)}')
    time.sleep(1.0)   # longer sleep to stay within rate limits

all_failed = already_failed | set(new_failures)
FAILED_FILE.write_text('\n'.join(sorted(all_failed)))
cached_count = len(list(PRICE_DIR.glob('*.parquet')))
print(f'\nDone. Cached: {cached_count:,}  |  Failed: {len(new_failures)}  |  Total failed ever: {len(all_failed)}')


$000001: possibly delisted; no timezone found


$000066: possibly delisted; no timezone found


$000060: possibly delisted; no timezone found


$000333: possibly delisted; no timezone found


$000050: possibly delisted; no timezone found


...

['ZUN.', 'ZWIPE']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


['ZPP.', 'ZYDUSWELL', 'ZURN', 'ZYDUSLIFE', 'ZXAIY']: possibly delisted; no timezone found



1 Failed download:


['ZZZ.']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')



Done. Cached: 5,624  |  Failed: 6087  |  Total failed ever: 9472


## 2.7 Forward Return Computation

For each event row we compute forward returns at horizons **1, 3, 5, 10, 20 trading days** from `entry_date`.

```
return_Nd = (close[entry_date + N_bday] / close[entry_date]) - 1
```

**Audit rule:** forward returns are stored as *target columns only* — never fed back as features.
The augmented parquet has a clear column naming convention (`fwd_1d`, `fwd_3d`, etc.) to make accidental
feature leakage easy to spot in code review.

In [10]:
HORIZONS = [1, 3, 5, 10, 20]

def load_price_series(ticker: str) -> pd.Series:
    """Load cached close prices for a ticker; returns empty Series on failure."""
    f = PRICE_DIR / f'{ticker}.parquet'
    if not f.exists():
        return pd.Series(dtype=float)
    p = pd.read_parquet(f).set_index('date')['close']
    p.index = pd.to_datetime(p.index).normalize()
    return p


def compute_forward_returns(price_series: pd.Series, entry_date: pd.Timestamp,
                             horizons: list) -> dict:
    """
    Compute close-to-close returns for each horizon from entry_date.

    If the price is missing on entry_date (e.g. holiday), roll forward up to 3
    business days to find a fill price.  The exit is computed from the ACTUAL
    fill date (not the original requested date) so that a 20d return is always
    20 trading days — never 17-19d due to a rolled entry.
    """
    result = {f'fwd_{h}d': np.nan for h in horizons}
    if price_series.empty:
        return result

    # Determine actual fill date
    actual_entry = entry_date
    entry_price  = price_series.get(entry_date, np.nan)
    if np.isnan(entry_price) or entry_price == 0:
        for offset in [1, 2, 3]:
            candidate   = entry_date + pd.offsets.BDay(offset)
            ep          = price_series.get(candidate, np.nan)
            if not np.isnan(ep) and ep != 0:
                entry_price  = ep
                actual_entry = candidate   # exit is measured from actual fill, not original date
                break

    if np.isnan(entry_price) or entry_price == 0:
        return result

    for h in horizons:
        exit_date  = actual_entry + pd.offsets.BDay(h)   # always h days from actual entry
        exit_price = price_series.get(exit_date, np.nan)
        if not np.isnan(exit_price) and exit_price != 0:
            result[f'fwd_{h}d'] = exit_price / entry_price - 1
    return result


# Compute returns per unique (ticker, entry_date) pair then merge back
pairs = df_total[['BESTTICKER','entry_date']].drop_duplicates().copy()
print(f'Computing forward returns for {len(pairs):,} unique (ticker, date) pairs...')

return_records = []
price_cache    = {}

for _, row in pairs.iterrows():
    t = row['BESTTICKER']
    d = row['entry_date']
    if t not in price_cache:
        price_cache[t] = load_price_series(t)
        if len(price_cache) > 500:
            oldest = next(iter(price_cache))
            del price_cache[oldest]
    rets = compute_forward_returns(price_cache[t], d, HORIZONS)
    rets['BESTTICKER'] = t
    rets['entry_date'] = d
    return_records.append(rets)

returns_df = pd.DataFrame(return_records)
df_total   = df_total.merge(returns_df, on=['BESTTICKER','entry_date'], how='left')

fwd_cols = [f'fwd_{h}d' for h in HORIZONS]
coverage  = df_total[fwd_cols].notna().mean()
print('\nForward return coverage (% of events with valid price):')
print(coverage.round(3).to_string())


Computing forward returns for 376,351 unique (ticker, date) pairs...



Forward return coverage (% of events with valid price):
fwd_1d     0.434
fwd_3d     0.437
fwd_5d     0.439
fwd_10d    0.433
fwd_20d    0.426


## 2.8 Save the Augmented Signal File

Write the Total slice enriched with `entry_date`, universe flags, and forward returns to Parquet.  
Downstream notebooks load this file — it is the single source of truth for the backtest.

In [11]:
df_total.to_parquet(AUGMENTED_PQ, index=False, engine='pyarrow')
print(f'Saved augmented signal file → {AUGMENTED_PQ}')
print(f'Shape: {df_total.shape}')

# Quick summary of what we have
print('\n=== Ready for backtesting ===')
for univ in ['in_sp500','in_sp1500','in_ru3k']:
    sub = df_total[df_total[univ]]
    cov = sub[fwd_cols].notna().mean()
    print(f"\n{univ.upper()}  ({len(sub):,} events, {sub['BESTTICKER'].nunique():,} tickers)")
    print(cov.round(3).to_string())

Saved augmented signal file → /Users/chaithanyapakala/Downloads/NLP_FinalProject/data/signals_with_returns.parquet
Shape: (376790, 610)

=== Ready for backtesting ===

IN_SP500  (24,041 events, 8,309 tickers)
fwd_1d     0.489
fwd_3d     0.494
fwd_5d     0.497
fwd_10d    0.494
fwd_20d    0.478



IN_SP1500  (49,809 events, 11,444 tickers)
fwd_1d     0.486
fwd_3d     0.490
fwd_5d     0.491
fwd_10d    0.487
fwd_20d    0.472

IN_RU3K  (29,538 events, 10,995 tickers)
fwd_1d     0.433
fwd_3d     0.438
fwd_5d     0.439
fwd_10d    0.434
fwd_20d    0.426
